# Agent results analysis

For each sampled repository we ran three agents on the fix commit and compared
their answer against the ground truth (the real CVE and CWE from CVEfixes).
This notebook scores the three agents, per agent and per repository.

**Agents**
- **agent1** sees only the code before, code after and diff of the commit.
- **agent2** sees the same code and can also navigate the repository.
- **agent3** sees only the repository, with no diff (zero-day style).

**Important caveat.** The ground truth is made of *positives only*: every
sampled commit fixes a real vulnerability. So on the found / not-found axis we
can only measure **recall** (does the agent detect the vulnerability that is
really there). There are no true negatives, so precision, specificity and the
false-positive rate cannot be computed here. The other axis we score is the
**CWE classification**: when an agent detects a vulnerability, does it assign
the correct CWE.

The run analysed below may be partial (some agent calls fail on very large
commits that exceed the model context, or hit rate limits); those show up as
*no answer* and are reflected in the coverage metric.

In [ ]:
import json
import re
import csv
import collections
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Which model's run to analyse: the folder name under outputs/ that main.py
# creates (a filesystem-safe version of the model id). Change MODEL to score a
# different model's run.
MODEL = 'openai/gpt-oss-20b:free'
MODEL_SLUG = re.sub(r'[^A-Za-z0-9._-]+', '_', MODEL)

GT_PATH = Path('ground_truth.csv')
RESULTS_PATH = Path('outputs') / MODEL_SLUG / 'agent_responses.jsonl'
AGENTS = ['agent1', 'agent2', 'agent3']

print('Model  :', MODEL)
print('Results:', RESULTS_PATH)

In [ ]:
# The verdict block every agent prompt ends with (see common.VERDICT_FORMAT),
# parsed here inline so this notebook depends on nothing from the pipeline.
def _extract_field(text, field):
    m = re.search(rf'\**{field}\**\s*:\s*\**\s*([^\n*]+)', text or '', re.IGNORECASE)
    return m.group(1).strip() if m else None

def parse_verdict(text):
    """(found, cwe_id) from an agent response. found is None when the answer
    has no parseable verdict (empty, or the model ignored the format)."""
    if not text:
        return None, None
    found_str = _extract_field(text, 'VULNERABILITY_FOUND')
    cwe = _extract_field(text, 'CWE_ID')
    found = found_str.strip().lower().startswith('yes') if found_str else None
    if cwe and cwe.strip().lower() == 'none':
        cwe = None
    return found, cwe

In [ ]:
# Ground truth: one entry per commit; gt_cwes is the UNION of all its CWE ids
# (a commit can fix several CVEs, each carrying one or more CWEs).
gt_cwes, gt_repo = {}, {}
for r in csv.DictReader(open(GT_PATH)):
    key = (r['repo_url'], r['commit'])
    cwes = {c.strip().upper() for c in (r['cwe_ids'] or '').split(',') if c.strip()}
    gt_cwes.setdefault(key, set()).update(cwes)
    gt_repo[key] = r['repo_name']

# Agent responses: keep the LAST record per (repo, commit, agent). The raw log
# appends a line on every retry, so the last one is the final outcome.
final = {}
for line in open(RESULTS_PATH):
    rec = json.loads(line)
    final[(rec['repo_url'], rec['commit'], rec['agent'])] = rec

N = len(gt_cwes)
print(f'{N} ground-truth commits, '
      f'{len({(k[0], k[1]) for k in final})} commits with at least one agent result')

In [ ]:
# Four mutually exclusive outcomes for one agent on one commit.
OUTCOMES = ['correct_cwe', 'detected_wrong_cwe', 'not_detected', 'no_answer']
OUTCOME_LABEL = {
    'correct_cwe':        'Correct CWE',
    'detected_wrong_cwe': 'Detected, wrong CWE',
    'not_detected':       'Not detected (said "no")',
    'no_answer':          'No answer (error / missing)',
}

def classify(rec, gt):
    """Outcome and predicted CWE of one agent on one commit."""
    if rec is None or rec['status'] != 'ok':
        return 'no_answer', None
    found, cwe = parse_verdict(rec.get('response'))
    if found is None:
        return 'no_answer', cwe
    if found is False:
        return 'not_detected', cwe
    if cwe and cwe.strip().upper() in gt:
        return 'correct_cwe', cwe
    return 'detected_wrong_cwe', cwe

rows = []
for key, gt in gt_cwes.items():
    for agent in AGENTS:
        outcome, pred = classify(final.get((key[0], key[1], agent)), gt)
        rows.append({'repo': gt_repo[key], 'commit': key[1][:9], 'agent': agent,
                     'outcome': outcome, 'pred_cwe': pred or '',
                     'gt_cwe': ', '.join(sorted(gt))})

df = pd.DataFrame(rows)
df.head()

## Metrics per agent

- **Coverage** the fraction of commits the agent actually answered (the rest
  errored out or were not reached).
- **Detection recall** among the answered commits, how often the agent said
  the vulnerability is there. Since every commit really is vulnerable, this is
  recall / sensitivity.
- **CWE accuracy (of detected)** among the commits it detected, how often the
  CWE it assigned matches the ground truth.
- **End to end** over all commits, how often the agent both detected the
  vulnerability and gave the correct CWE.

In [ ]:
recs = []
for agent in AGENTS:
    c = df[df.agent == agent].outcome.value_counts().to_dict()
    correct = c.get('correct_cwe', 0)
    wrong   = c.get('detected_wrong_cwe', 0)
    notdet  = c.get('not_detected', 0)
    noans   = c.get('no_answer', 0)
    answered = N - noans
    detected = correct + wrong
    recs.append({
        'agent': agent,
        'coverage':         answered / N,
        'detection_recall': detected / answered if answered else np.nan,
        'cwe_acc_detected': correct / detected if detected else np.nan,
        'end_to_end':       correct / N,
        'correct': correct, 'wrong_cwe': wrong,
        'not_detected': notdet, 'no_answer': noans,
    })

metrics = pd.DataFrame(recs).set_index('agent')
rate_cols = ['coverage', 'detection_recall', 'cwe_acc_detected', 'end_to_end']
metrics.style.format({c: '{:.0%}' for c in rate_cols})

## How each agent's commits break down

In [ ]:
OUTCOME_COLOR = {
    'correct_cwe':        '#2a78d6',   # blue
    'detected_wrong_cwe': '#eb6834',   # orange
    'not_detected':       '#1baf7a',   # aqua
    'no_answer':          '#b0aea4',   # neutral grey
}

counts = (df.groupby(['agent', 'outcome']).size().unstack(fill_value=0)
            .reindex(columns=OUTCOMES, fill_value=0).reindex(AGENTS))

fig, ax = plt.subplots(figsize=(8.5, 3.6))
y = np.arange(len(AGENTS))
left = np.zeros(len(AGENTS))
for oc in OUTCOMES:
    vals = counts[oc].to_numpy()
    ax.barh(y, vals, left=left, color=OUTCOME_COLOR[oc],
            label=OUTCOME_LABEL[oc], edgecolor='white', linewidth=1.5)
    for yi, (v, l) in enumerate(zip(vals, left)):
        if v > 0:
            ax.text(l + v / 2, yi, int(v), ha='center', va='center',
                    color='white', fontweight='bold', fontsize=9)
    left += vals

ax.set_yticks(y)
ax.set_yticklabels(AGENTS)
ax.invert_yaxis()
ax.set_xlabel(f'Commits (out of {N})')
ax.set_title('Outcome of each agent across the sampled commits')
ax.legend(frameon=False, bbox_to_anchor=(1.01, 1.0), loc='upper left')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
show = ['coverage', 'detection_recall', 'cwe_acc_detected', 'end_to_end']
labels = ['Coverage', 'Detection\nrecall', 'CWE acc.\n(of detected)', 'End to end\n(correct CWE)']
AGENT_COLOR = {'agent1': '#2a78d6', 'agent2': '#eb6834', 'agent3': '#1baf7a'}

x = np.arange(len(show))
w = 0.26
fig, ax = plt.subplots(figsize=(9, 4.2))
for i, agent in enumerate(AGENTS):
    vals = [metrics.loc[agent, m] for m in show]
    bars = ax.bar(x + (i - 1) * w, vals, w, color=AGENT_COLOR[agent],
                  label=agent, edgecolor='white', linewidth=1)
    for b, v in zip(bars, vals):
        if pd.notna(v):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f'{v:.0%}',
                    ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.08)
ax.set_ylabel('Rate')
ax.yaxis.set_major_formatter(lambda v, _: f'{v:.0%}')
ax.set_title('Key metrics per agent')
ax.legend(frameon=False, title='')
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## Per repository

The same result, one row per repository, showing what each agent concluded and
the CWE it assigned, next to the ground-truth CWE.

In [ ]:
MARK = {'correct_cwe': 'correct', 'detected_wrong_cwe': 'wrong-cwe',
        'not_detected': 'missed', 'no_answer': 'no-answer'}

def cell(o, pred):
    return MARK[o] + (f' [{pred}]' if pred else '')

detail = df.copy()
detail['cell'] = [cell(o, p) for o, p in zip(detail.outcome, detail.pred_cwe)]
per_repo = (detail.pivot_table(index=['repo', 'gt_cwe'], columns='agent',
                               values='cell', aggfunc='first')
                  .reindex(columns=AGENTS))
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(per_repo)

In [ ]:
# Heatmap: rows = repositories, columns = agents, colour = outcome,
# annotated with the predicted CWE number (relief rule: never colour alone).
code_of = {oc: i for i, oc in enumerate(OUTCOMES)}
outcome_mat = (df.pivot_table(index='repo', columns='agent', values='outcome',
                              aggfunc='first').reindex(columns=AGENTS))
pred_mat = (df.pivot_table(index='repo', columns='agent', values='pred_cwe',
                           aggfunc='first').reindex(columns=AGENTS))

repos = list(outcome_mat.index)
M = np.array([[code_of.get(outcome_mat.loc[r, a], len(OUTCOMES) - 1)
               for a in AGENTS] for r in repos])
cmap = ListedColormap([OUTCOME_COLOR[o] for o in OUTCOMES])

fig, ax = plt.subplots(figsize=(6.5, max(4, 0.34 * len(repos))))
ax.imshow(M, cmap=cmap, vmin=0, vmax=len(OUTCOMES) - 1, aspect='auto')
ax.set_xticks(range(len(AGENTS)))
ax.set_xticklabels(AGENTS)
ax.set_yticks(range(len(repos)))
ax.set_yticklabels(repos, fontsize=7)
for i, r in enumerate(repos):
    for j, a in enumerate(AGENTS):
        p = pred_mat.loc[r, a]
        if isinstance(p, str) and p:
            ax.text(j, i, p.replace('CWE-', ''), ha='center', va='center',
                    fontsize=6, color='white', fontweight='bold')
ax.set_title('Per-repo outcome (colour) and predicted CWE (number)')
ax.legend(handles=[Patch(color=OUTCOME_COLOR[o], label=OUTCOME_LABEL[o])
                   for o in OUTCOMES],
          bbox_to_anchor=(1.02, 1.0), loc='upper left', frameon=False)
plt.tight_layout()
plt.show()

## What the numbers say

Read this against the run that is actually loaded above (the figures update
when you re-run on fresh results).

- **CWE classification is the hard part, not detection.** agent1 and agent2
  almost always say a vulnerability is present when they answer, which is easy
  here because every commit really is a fix, but they assign the correct CWE
  only about half the time.
- **agent3 (zero-day) is much weaker.** With no diff to point at the change, it
  rarely locates the vulnerability and almost never classifies it, which is the
  motivation for exploring a graph representation of the repository to guide it.
- **Coverage differs by design.** agent1 and agent2 send the full before / after
  / diff, so on very large commits the prompt exceeds the model context and the
  call fails (a smaller-model limitation); agent3 sends no code, so it fails
  less often on that account.

Because the ground truth has no true negatives, none of this measures false
positives: an agent that cried "vulnerability" on everything would score the
same recall. Adding negative examples (see the discussion on external
non-vulnerable sources) is what would let us compute precision and F1.